# debug_inference
起動・デバッグ用のノートブック。処理の本体は `app/`（アプリ）と `scripts/`（成果物の生成）にあり、ここでは呼び出すだけ。

コマンド1つで起動したい場合は `python run.py`（`--tunnel` で外部公開、`--rag` で RAG を有効化）でも同じことができる。

**実行する場所**：このノートブックは `backend/` を作業フォルダとして実行する（研究室サーバなどでは、`backend/` で開けばそのまま動く）。

## 準備（Google Colab の場合のみ）
Google Colab では、セッションのたびに環境がまっさらになるので、下の2つのセルを毎回実行する。研究室サーバなど手元の環境では、初回にライブラリと VOICEVOX を入れてあれば不要。

In [ ]:
# ## Google Colab：Google Drive をマウントし、backend/ に移動する
# from google.colab import drive
# drive.mount("/content/drive/")
# %cd "/content/drive/MyDrive/Colab Notebooks/app_labGuide_v2/backend"

In [ ]:
# ## このセルはGoogle Colabで実行するとき：ライブラリと VOICEVOX の準備
# # 1. VOICEVOX エンジンと cloudflared の導入（tools/ にダウンロードされる。既にあれば省略される）
# !bash ../setup.sh

# # 2. Python ライブラリのインストール
# !pip install -q -r ../requirements.txt

# # 3. VOICEVOX をローカル（/content）にコピーして、実行権限を付ける
# #    Google Drive 上のファイルには実行権限を付けられないため、Colab ではコピーして使う
# #    （/content はセッションが終わると消えるので、毎回実行する）
# !rm -rf /content/voicevox && cp -r ../tools/linux-cpu-x64 /content/voicevox
# !chmod +x /content/voicevox/run

## RAG の設定（任意）
RAG を使うと、キャラクター設定（`data/raw/rag_docs/kokaFumi.txt`）を検索して返答の参考情報にする。既定では使わない。

使う場合は、`USE_RAG = True` にして、初回（と文書を変えたとき）に検索用インデックスを作る。インデックスの作成には、埋め込みモデル（約2GB）のダウンロードが必要。

In [ ]:
USE_RAG = False      # RAG を使うなら True
# USE_RAG = True

if USE_RAG:
    # data/raw/rag_docs/ の .txt から、検索用インデックス（data/processed/rag_store/）を作る
    !python app/rag/build_index.py

## 設定とモデルのロード
設定値（モデル・層・成果物の場所・量子化など）は `run.py` の先頭にまとめてある。変える場合はそちらを編集する。

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())   # 今いる backend/ を、モジュールを探す場所に加える

import run   # backend/ に移動し、sys.path を設定する
from run import (MODEL_ID, RESEARCH_MODEL_KEY, TARGET_LAYER, ASSETS_ROOT, QUANTIZATION_8BIT,
                 build_manager, start_voicevox, start_server)
from app.core.llm_engine import LLMEngine

IN_COLAB = "google.colab" in sys.modules   # Google Colab で動いているか

print("model :", MODEL_ID)
print("layer :", TARGET_LAYER, "(hidden_states のインデックス)")
print("assets:", ASSETS_ROOT)
print("colab :", IN_COLAB, " / RAG:", USE_RAG)

In [ ]:
# 量子化するかは run.py の QUANTIZATION_8BIT で決める（既定は False）
# 量子化すると活性化の値が変わり、成果物の avg_norm・α上限と対応しなくなるので、
# GPU メモリが足りないときの最終手段として使う
engine = LLMEngine(model_id=MODEL_ID, quantization_8bit=QUANTIZATION_8BIT)

## 研究成果物の作り直し（任意・通常は不要）
同梱の `data/processed/steering_assets/` をそのまま使うなら実行しなくてよい。

| ステップ | 内容 | 必要なもの |
|---|---|---|
| activations | 合成データの隠れ状態を抽出してキャッシュ | GPU・合成データ |
| vectors | CAA＋中立PCAで感情ステアリングベクトル | activations のキャッシュ |
| probe | 線形感情プローブ（8感情＋neutral） | activations のキャッシュ |
| refusal | 拒絶方向 r_hat | GPU・拒絶ペアCSV |

合成データと拒絶ペアは公開していないため、clone しただけでは実行できない（README の「研究成果物の再生成」を参照）。ロード済みの `engine` を使い回すので、モデルを二重に読み込まない。α上限はここでは作らない（研究リポジトリで決めた値を使う）。

In [ ]:
REBUILD_ASSETS = False
if REBUILD_ASSETS:
    from scripts.build_assets import build_all
    build_all(engine, model_key=RESEARCH_MODEL_KEY, layer=TARGET_LAYER,
              steps=["activations", "vectors", "probe"])   # "refusal" は拒絶ペアCSVがある場合のみ

## マネージャの作成

In [ ]:
manager = build_manager(engine, use_rag=USE_RAG)

### 動作確認（UIを使わずに試す）

In [ ]:
# 感情プローブだけを試す（内部状態は変わらない）
for t in ["この前話してた本、やっと読み終わったよ",
          "大学の学食について教えてください",
          "この対応は本当にひどい"]:
    print(t, "->", manager.classifier.predict_top(t, k=3),
          f"neutral={manager.classifier.neutral_score(t):.2f}")

In [ ]:
# 1ターン生成する（内部状態と履歴が更新される。終わったら reset する）
res = manager.generate_reply("この前話してた本、やっと読み終わったよ", condition="steering")
print(res["reply"])
print("emotion:", res["complex_emotion"], " alpha:", res["applied_alpha"])
manager.reset_conversation()

## サーバの起動
`tunnel=True` で、cloudflared の一時的な公開URLを発行する（URL を知っている人は誰でもアクセスできるので、使い終わったら止めること）。`tunnel=False` にすると http://localhost:8000 のみで起動する。

In [ ]:
# VOICEVOX：Colab では /content にコピーしたもの、それ以外は tools/ のものを使う
# （エンジンが無い・起動できない場合は None になり、音声なしで動く）
voicevox = start_voicevox("/content/voicevox" if IN_COLAB else run.VOICEVOX_DIR)

app = start_server(manager, voicevox, port=8000, tunnel=True, background=True, rag=USE_RAG)